# 🚀 실전 RAG 파이프라인 (FAISS + LLM)

In [1]:
import warnings
warnings.filterwarnings('ignore')

import faiss
import numpy as np
from sentence_transformers import SentenceTransformer
from transformers import pipeline

## 1. 지식 베이스(문서) 임베딩 및 FAISS 인덱스 빌드
소형 임베딩 모델을 사용해 영어 텍스트를 벡터로 바꾸고, 페이스북(Meta)이 만든 고속 벡터 검색기인 FAISS에 담습니다.

In [2]:
# 가상의 사내 기밀 지식 데이터셋
documents = [
    "Project Apollo is our secret plan to launch a new AI smartphone next year. The budget is $5B.",
    "The CEO of our company, Mr. Rockman, is planning to acquire a robotics firm in Tokyo.",
    "Our cafeteria menu for this Friday features special spicy chicken tacos and mango salsa.",
    "To access the internal VPN, employees must use the Cisco AnyConnect client and a physical OTP token."
]

# 임베딩 모델 로드 (HuggingFace)
print("임베딩 모델 로드 중...")
embedder = SentenceTransformer('all-MiniLM-L6-v2')

# 문서들을 384차원 Dense Vector로 변환
doc_embeddings = embedder.encode(documents)
print(f"문서 임베딩 완료! 차원 크기: {doc_embeddings.shape}")

# FAISS 벡터 인덱스 생성 (단순 L2 거리 기반 IndexFlatL2 사용, 대용량일 경우 IndexHNSW를 사용함)
dimension = doc_embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)
index.add(np.array(doc_embeddings, dtype=np.float32))

print(f"FAISS 벡터 DB에 {index.ntotal}개의 문서 적재 완료.")

임베딩 모델 로드 중...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

문서 임베딩 완료! 차원 크기: (4, 384)
FAISS 벡터 DB에 4개의 문서 적재 완료.


## 2. 사용자 질문 검색 (Retrieval)
질문이 들어오면 동일한 임베딩 모델로 벡터화한 뒤 FAISS를 통해 가장 유사한 상위 1개의 문서를 찾아옵니다.

In [3]:
user_question = "What is the budget for the new AI phone project?"

# 질문을 벡터로 변환
query_vector = embedder.encode([user_question])

# FAISS에서 Top-1 검색
k = 1
distances, indices = index.search(np.array(query_vector, dtype=np.float32), k)

retrieved_doc = documents[indices[0][0]]
print(f"사용자 질문: {user_question}")
print(f"\n🔍 [검색된 관련 문서]: {retrieved_doc}")

사용자 질문: What is the budget for the new AI phone project?

🔍 [검색된 관련 문서]: Project Apollo is our secret plan to launch a new AI smartphone next year. The budget is $5B.


## 3. 언어 모델을 활용한 답변 생성 (Generation)
검색된 문서를 프롬프트의 배경 지식(Context)으로 삽입하고, 소형 텍스트 생성 모델에게 질문에 답하도록 지시합니다.

In [4]:
print("언어 모델 로드 중...")
# 빠르고 가벼운 생성 모델 로드
generator = pipeline("text-generation", model="sshleifer/tiny-gpt2", max_length=100)

# RAG 프롬프트 합성 (Context + Question)
prompt = f"""Context information is below.
---------------------
{retrieved_doc}
---------------------
Given the context information, answer the question: {user_question}
Answer: """

print("\n[합성된 최종 프롬프트]")
print(prompt)

print("\n🤖 [LLM의 답변 생성 중...]")
response = generator(prompt, num_return_sequences=1)
print(response[0]['generated_text'])

print("\n💡 RAG 파이프라인 덕분에 모델이 환각(거짓말)을 하지 않고 실제 회사 기밀 정보를 기반으로 답변을 생성할 수 있는 구조가 완성되었습니다!")

언어 모델 로드 중...


Loading weights:   0%|          | 0/29 [00:00<?, ?it/s]

[transformers] GPT2LMHeadModel LOAD REPORT from: sshleifer/tiny-gpt2
Key                                   | Status     |  | 
--------------------------------------+------------+--+-
transformer.h.{0, 1}.attn.masked_bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


[transformers] Passing `generation_config` together with generation-related arguments=({'max_length'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


[transformers] Passing `generation_config` together with generation-related arguments=({'num_return_sequences'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


[transformers] Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.



[합성된 최종 프롬프트]
Context information is below.
---------------------
Project Apollo is our secret plan to launch a new AI smartphone next year. The budget is $5B.
---------------------
Given the context information, answer the question: What is the budget for the new AI phone project?
Answer: 

🤖 [LLM의 답변 생성 중...]


ValueError: Input length of input_ids is 59, but `max_length` is set to 50. This can lead to unexpected behavior. You should consider increasing `max_length` or, better yet, setting `max_new_tokens`.